# Training Script for ML Time Series Models

## Settings and Imports

In [1]:

from datetime import date
import pandas as pd
import os
import matplotlib.pyplot as plt

import torch
import torch.optim as optim
import torch.utils.data as data
import torch.nn as nn

import numpy as np

from utils.data_preparation import prepare_data_for_modeling, create_torch_dataset
from models.lstm_model import LSTMForecaster

os.makedirs("data", exist_ok=True)
os.makedirs("models", exist_ok=True)

BATCH_SIZE = 16
EPOCHS = 50000
LEARNING_RATE = 1e-3
WINDOW_SIZE = 7  # Number of days to look back for the LSTM model

if torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
elif torch.cuda.is_available():
    DEVICE = torch.device("cuda")
else:
    DEVICE = torch.device("cpu")
print(f"Using device: {DEVICE}")

# This features describe the today's weather and load. We will predict the load of the next day
FEATURES_TODAY = ['Temp', 'Min Temp', 'Max Temp', 'Wind Speed',
                  'Sunshine Duration', 'Cloud Cover', 'load_mw', 'lag_1',
                  'lag_7', 'lag_14', 'rolling_mean_7']

# These features describe the features of the next day (e.g. is it a weekend, holiday, etc.). We will predict the load of the next day
FEATURES_TARGET_TIME = ['is_weekend', 'is_holiday',
                        'dow_sin', 'dow_cos', 'month_sin', 'month_cos']

FEATURES = FEATURES_TODAY + FEATURES_TARGET_TIME

# We will predict the load of the next day
TARGET = "load_tomorrow"

SCALE_FEATURES = ['Temp', 'Min Temp', 'Max Temp', 'Wind Speed',
                  'Sunshine Duration', 'Cloud Cover', 'load_mw', 'lag_1',
                  'lag_7', 'lag_14', 'rolling_mean_7']

Using device: mps


## Dataset Loading and Preparation

In [2]:
data_dict, _ = prepare_data_for_modeling(FEATURES, TARGET, SCALE_FEATURES, save_scaler=True, save_data=True)

# Training Data
X_train_scaled = data_dict['X_train_scaled']
y_train = data_dict['y_train']
X_train_tensor, y_train_tensor = create_torch_dataset(X=X_train_scaled, y=y_train, window_size=WINDOW_SIZE)
training_loader = data.DataLoader(data.TensorDataset(X_train_tensor, y_train_tensor), shuffle=True, batch_size=BATCH_SIZE)

# Validation Data
X_val_scaled = data_dict['X_val_scaled']
y_val = data_dict['y_val']
X_val_tensor, y_val_tensor = create_torch_dataset(X=X_val_scaled, y=y_val, window_size=WINDOW_SIZE)
validation_loader = data.DataLoader(data.TensorDataset(X_val_tensor, y_val_tensor), shuffle=False, batch_size=BATCH_SIZE)

# Test Data
X_test_scaled = data_dict['X_test_scaled']
y_test = data_dict['y_test']
X_test_tensor, y_test_tensor = create_torch_dataset(X=X_test_scaled, y=y_test, window_size=WINDOW_SIZE)
test_loader = data.DataLoader(data.TensorDataset(X_test_tensor, y_test_tensor), shuffle=False, batch_size=BATCH_SIZE)

test_df = data_dict['test_df'] # For the timestamps

## Training Setup

In [3]:
model = LSTMForecaster(input_size=len(FEATURES), hidden_size=64, num_layers=2, output_size=1).to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
loss_fn = nn.MSELoss()
BEST_MODEL_PATH = os.path.join("models", "best_lstm_load_forecaster.pt")


## Training

In [4]:
best_val_mse = float("inf")
best_epoch = -1

for epoch in range(EPOCHS):
    model.train()
    train_mse_sum = 0.0
    train_mae_sum = 0.0
    train_samples = 0

    for X_batch, y_batch in training_loader:
        X_batch = X_batch.to(DEVICE)
        y_batch = y_batch.to(DEVICE)
        y_pred = model(X_batch).squeeze(-1)
        loss = loss_fn(y_pred, y_batch)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        train_samples += batch_size
        train_mse_sum += torch.sum((y_pred - y_batch) ** 2).item()
        train_mae_sum += torch.sum(torch.abs(y_pred - y_batch)).item()

    if epoch % 100 != 0:
        continue

    model.eval()
    val_mse_sum = 0.0
    val_mae_sum = 0.0
    val_samples = 0

    with torch.no_grad():
        for X_batch, y_batch in validation_loader:
            X_batch = X_batch.to(DEVICE)
            y_batch = y_batch.to(DEVICE)
            y_pred = model(X_batch).squeeze(-1)
            batch_size = y_batch.size(0)
            val_samples += batch_size
            val_mse_sum += torch.sum((y_pred - y_batch) ** 2).item()
            val_mae_sum += torch.sum(torch.abs(y_pred - y_batch)).item()

    train_mse = train_mse_sum / train_samples
    train_rmse = np.sqrt(train_mse)
    train_mae = train_mae_sum / train_samples
    val_mse = val_mse_sum / val_samples
    val_rmse = np.sqrt(val_mse)
    val_mae = val_mae_sum / val_samples

    if val_mse < best_val_mse:
        best_val_mse = val_mse
        best_epoch = epoch
        torch.save(model.state_dict(), BEST_MODEL_PATH)
        checkpoint_note = " [saved best]"
    else:
        checkpoint_note = ""

    print(
        f"Epoch {epoch}: train RMSE {train_rmse:.4f}, train MAE {train_mae:.4f}, "
        f"val RMSE {val_rmse:.4f}, val MAE {val_mae:.4f}, val MSE {val_mse:.4f}{checkpoint_note}"
    )

print(f"Best model saved to {BEST_MODEL_PATH} (epoch {best_epoch}, val MSE {best_val_mse:.6f})")

Epoch 0: train RMSE 56207.8286, train MAE 55798.3954, val RMSE 52946.2400, val MAE 52605.1870, val MSE 2803304335.2116 [saved best]
Epoch 100: train RMSE 55331.2521, train MAE 54915.2857, val RMSE 52070.6376, val MAE 51723.8117, val MSE 2711351296.0000 [saved best]
Epoch 200: train RMSE 54455.4670, train MAE 54032.7605, val RMSE 51194.0932, val MAE 50841.2882, val MSE 2620835175.1420 [saved best]
Epoch 300: train RMSE 53579.9322, train MAE 53150.2603, val RMSE 50317.7812, val MAE 49958.7882, val MSE 2531879107.8957 [saved best]
Epoch 400: train RMSE 52704.6758, train MAE 52267.8017, val RMSE 49441.7322, val MAE 49076.3310, val MSE 2444484881.0667 [saved best]
Epoch 500: train RMSE 51829.7019, train MAE 51385.3999, val RMSE 48565.9690, val MAE 48193.9289, val MSE 2358653343.5362 [saved best]
Epoch 600: train RMSE 50954.9859, train MAE 50502.9740, val RMSE 47690.4392, val MAE 47311.5149, val MSE 2274377989.1942 [saved best]
Epoch 700: train RMSE 50080.5952, train MAE 49620.6355, val RMSE